In [2]:
####IMPORTANTE CARGAR UTILS.PY
from google.colab import files
uploaded = files.upload()

Saving utils.py to utils.py


In [3]:
!pip install optuna
!pip install -U kaleido
!pip install optuna-dashboard
#!pip install kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.2/104.2 kB 13.4 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd

from sklearn.metrics import cohen_kappa_score, accuracy_score,balanced_accuracy_score

from plotly import express as px

from utils import plot_confusion_matrix, get_artifact_filename

import os

from json import loads

from joblib import load, dump

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

In [17]:
#Importar json de kaggle
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"sebastianayalasosa","key":"726ac384b3a464b0f04f156dceeef26c"}'}

In [19]:
import os
import zipfile

# Crear la carpeta .kaggle y mover el archivo
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [20]:
!kaggle competitions download -c petfinder-adoption-prediction

In [21]:
!unzip -q petfinder-adoption-prediction.zip -d input/petfinder-adoption-prediction

replace input/petfinder-adoption-prediction/train/train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y


In [22]:
BASE_DIR = './'  # ya que estás en Colab
PATH_TO_TRAIN = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train/train.csv")
PATH_TO_IMAGES_DIR = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train_images")

In [23]:
# Paths
BASE_DIR = './'  # ya que estás en Colab
PATH_TO_MODELS = os.path.join(BASE_DIR, "work/models")
PATH_TO_TRAIN = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train/train.csv")
PATH_TO_IMAGES_DIR = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train_images")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")


In [25]:
# study_lgb = optuna.create_study(direction='maximize',
#                             storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
#                             study_name="04 - LGB Multiclass CV",
#                             load_if_exists = True)

study_lgb = optuna.create_study(
    direction='maximize',
    storage="sqlite:///work/db.sqlite3",
    study_name="04 - LGB Multiclass CV",
    load_if_exists=True
)

lgb_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_lgb,'test')))

[I 2025-04-25 22:33:15,619] Using an existing study with name '04 - LGB Multiclass CV' instead of creating a new one.


In [26]:
lgb_dataset

,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed,pred
14696,1,Dione & Elora,1,307,307,2,1,0,0,2,...,2,0,41327,61b07b54adb97d4b5f3c2dec06a9943b,0,Dione and Elora are puppies of Rambo. Both are...,8f20e24ef,9.0,4,"[0.06903981805409481, 0.8420570745901301, 1.87..."
14823,1,Har-nee,24,103,307,2,1,2,4,2,...,1,0,41330,9cb2e5a10e24e0b09942013b8434c81f,0,We found Har-nee with a swollen and almost sev...,2d72ef0c4,2.0,4,"[0.10805223804114165, 0.9297631160831572, 1.11..."
2838,1,The Gorgeous 5 Beauties,2,307,0,2,2,7,0,2,...,5,0,41326,5c398b2e18b16f0db83c53e682eada42,0,Theses 5 very adorably cute white female puppi...,44cd12263,5.0,4,"[0.06366986416594626, 0.5421884405741726, 1.65..."
1848,2,Mochi,1,265,0,1,2,0,0,1,...,1,0,41401,6905e4fbe5658eef5f560b814898a5ee,2,Hello! My name is Mochi. I was rescued from a ...,210c4a637,6.0,2,"[0.1491433836393099, 1.2007799589538517, 2.041..."
669,2,Nala & Peach,9,266,266,2,2,4,6,2,...,2,0,41326,803457cd3660dda694086b51a11a5a39,0,Nala is a cat that's been born with 7 fingers ...,21493e6ea,8.0,4,"[0.0999169828432316, 0.6455706391645725, 1.672..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,2,Anak Nanya,8,266,0,3,1,2,0,2,...,3,50,41326,f14c2cfebbbafbc9ed1f500d082f3ec3,0,they r ol so cute :) it juz a matter me dun hv...,35f9818a7,14.0,4,"[0.06309989409593196, 0.7152185499096019, 1.31..."
12222,1,Poor Baby,3,307,0,1,5,0,0,2,...,1,0,41401,500c48db7b281eabec3c293160f4a71c,0,On behalf of Exotica Pets Healthy puppy availa...,46e25aa2b,2.0,1,"[0.0663570918696692, 1.3766785631975564, 1.459..."
10538,2,No Name,1,265,0,2,1,6,0,1,...,1,0,41401,ac9a633cf51a70f4a9842e6e1ba91fc9,0,sy jumpa kitten ni mengiau2 kat playground. ra...,d3692d2b2,2.0,1,"[0.18792248354416807, 1.8819909139752338, 1.53..."
11062,1,Pipi,1,307,0,2,1,5,7,2,...,6,0,41326,3ef66c1034bb6dc31314845457079483,0,"Health, cute and active puppies.",3c43b7541,1.0,4,"[0.07565913185903282, 0.9859051470375686, 1.55..."


In [28]:
MODEL_NAME = '04 ResNet'
MODEL_VERSION = '1.0.0'

# study_resnet = optuna.create_study(direction='maximize',
#                             storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
#                             study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
#                             load_if_exists = True)

study_resnet = optuna.create_study(
    direction='maximize',
    storage="sqlite:///work/db1.sqlite3",
    study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
    load_if_exists=True
)


resnet_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_resnet,'test')))

[I 2025-04-25 22:39:08,178] Using an existing study with name '04 ResNet_1.0.0' instead of creating a new one.


In [29]:
resnet_dataset

,PetID,pred,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,...,Sterilized,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PhotoAmt,AdoptionSpeed
0,015da9e87,"[-1.011046, 0.55211455, 0.34312776, -0.0169732...",2,Adik Gebuk (Betina),2,265,266,2,2,5,...,2,1,1,0,41326,d718a8deb57887c6ee18b757484273c8,0,Nama: Gebuk (Betina)- Adik beradik dengan Gebu...,5.0,0
1,022606901,"[-3.1871738, -0.5187237, 1.1013533, 0.95930845...",1,NaN,3,141,307,1,1,0,...,2,1,1,0,41401,c4b8b921e00ba5dc19e793b81987f40f,1,Hi all =) My friend is currently looking for s...,5.0,0
2,02f89bdcb,"[-2.1278024, 0.12823644, 0.34953502, 0.887814,...",1,Rex,72,141,0,1,5,0,...,2,1,1,0,41326,e76b700e2c869088979aa5efeb962dd7,0,Friendly and playful. Good watchdog because of...,3.0,0
3,0cf7fae9d,"[-1.1634104, 0.6626059, 0.19420704, 0.02009970...",2,KITTENS - URGENT ADOPTION,1,266,0,3,1,2,...,2,1,4,0,41326,1eea485b01d14c668f33afa7c919646e,0,These 4 kittens need urgent adoption because t...,1.0,0
4,0e922caab,"[-0.96127534, 1.0843039, 0.42076063, 0.2070416...",1,Ha Ha (Toy Poodle),12,179,0,1,2,0,...,2,1,1,300,41326,225d19c861c7c5d20a9c3ba1b2d37753,0,Ha Ha belongs to my friend who migrated to ano...,5.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2926,ff2cf88a0,"[-2.54497, 0.019052565, 0.77428615, 1.01089, 1...",1,JACKSON,12,307,0,1,2,0,...,1,1,1,0,41326,2266840747a7440f9f3453e31b384df5,0,Meet Jackson. He's a eye catcher and always re...,2.0,4
2927,ff498c903,"[-1.9024731, 0.5495223, 0.5196323, 0.9043485, ...",1,Lady,63,189,307,2,1,2,...,1,1,1,0,41326,03806ca295ace12b7463f4ed036cbb0e,0,Lady was an abandoned dog last time after she ...,5.0,4
2928,ff50c6171,"[-1.3058587, 0.5527345, 0.21869129, 0.17640659...",2,Gari,4,247,0,2,1,2,...,2,1,1,50,41326,2ca58d9cdf6107e7169985db6562bc3e,0,House kitten. Malaysian friend gave to me. but...,5.0,4
2929,ff5e30380,"[-1.6898106, 0.2825857, 0.34715194, 0.11564552...",2,Fa Meow,4,254,0,2,1,3,...,1,1,1,100,41401,1faf0ae111772205cf2f28b3ecea3276,0,"Long haired like persian cat, friendly, fast r...",5.0,4


In [30]:
merged_datasets = lgb_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred':'lgb_pred_score'},axis=1).merge(resnet_dataset[['PetID', 'pred']].rename({'pred':'resnet_pred_score'},axis=1),
                  on='PetID', how='outer')



merged_datasets['resnet_pred_score'] = [np.zeros(5) if type(i) is float else  i for i in merged_datasets['resnet_pred_score'] ]

In [31]:
merged_datasets['resnet_pred_score']

,resnet_pred_score
0,"[-1.5438033, 0.52494735, 0.44315472, 0.0745236..."
1,"[-1.5291157, 0.94259745, 0.13885194, 0.1639244..."
2,"[-1.3775858, 0.9968878, 1.2852389, 0.21161136,..."
3,"[-2.5076375, -0.06754442, 0.45959297, 0.453150..."
4,"[-2.337191, 0.69180363, 0.79296255, 0.6912336,..."
...,...
2994,"[-1.9906414, 0.5501437, 0.49609032, 0.03704826..."
2995,"[-1.7771401, -0.16564259, 0.7572702, 0.4501834..."
2996,"[-2.593443, 0.6291706, 1.2185475, 0.71821684, ..."
2997,"[-1.7730166, -0.19435912, 0.553412, -0.1415244..."


In [32]:
merged_datasets['blend_pred_score'] = [r['lgb_pred_score']+r['resnet_pred_score'] for i,r in merged_datasets.iterrows()]

In [42]:
# prompt: Gemini quiero una tabla dinamica de la primera fila del data set merged_datasets, ya que quiero ver todos los digitos y los valores de la primera fila

import pandas as pd

# Assuming 'merged_datasets' is already defined as in your provided code.

# Display the first row as a DataFrame for better formatting
first_row_df = pd.DataFrame(merged_datasets.iloc[0]).transpose()
display(first_row_df)


,PetID,lgb_pred_score,AdoptionSpeed,resnet_pred_score,blend_pred_score,lgb_pred,resnet_pred,blended_pred
0,002230dea,"[0.19829921494904373, 1.505388937919952, 1.878...",1,"[-1.5438033, 0.52494735, 0.44315472, 0.0745236...","[-1.3455041192871013, 2.0303362831767577, 2.32...",2,1,2


In [41]:
merged_datasets.iloc[0,].T

,0
PetID,002230dea
lgb_pred_score,"[0.19829921494904373, 1.505388937919952, 1.878..."
AdoptionSpeed,1
resnet_pred_score,"[-1.5438033, 0.52494735, 0.44315472, 0.0745236..."
blend_pred_score,"[-1.3455041192871013, 2.0303362831767577, 2.32..."
lgb_pred,2
resnet_pred,1
blended_pred,2


In [33]:
merged_datasets['lgb_pred'] = [r.argmax() for r in merged_datasets['lgb_pred_score']]
merged_datasets['resnet_pred'] = [r.argmax() for r in merged_datasets['resnet_pred_score']]
merged_datasets['blended_pred'] = [r.argmax() for r in merged_datasets['blend_pred_score']]

In [34]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['lgb_pred'],
                    title = 'LGB Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['lgb_pred'],
                                                                    weights='quadratic')))

In [35]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['resnet_pred'],
                    title = 'Resnet Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['resnet_pred'],
                                                                    weights='quadratic')))



In [36]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['blended_pred'],
                    title = 'Blended Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['blended_pred'],
                                                                    weights='quadratic')))
